# Session analysis, hand rehabilitation device

Basil Toufexis, MXEN4000 / MXEN4004, Curtin University.

Pick a save from the dropdown, then run the cells under it **in order**. Each
cell does one thing and prints its own result directly beneath, so the working
stays visible instead of arriving as one block from a single function.

A **game** is one folder, one block of one mode. Games by the same person on the
same day make up a **session**.

**Run the cells in order after changing the dropdown.** Every cell below checks
that it is looking at the save the dropdown currently names, and stops with a
"re-run the cells above" message rather than blending two selections into one
headline table.

Force needs a warning up front. Each sensor pad reads a different number of
counts for the same real force, so raw counts are not comparable between
fingers. Where a session recorded a calibration, force is also given as a
fraction of the light press that finger gave at calibration. That fraction is
**not** a strength ranking: the reference press is the patient's own press, so
a weak finger recorded a small reference and dividing by it cancels the
weakness along with the pad difference. Read it as effort against that finger's
own reference. For "which finger is stronger", read the newton column and
accept that the pad differences are not corrected there. This device cannot
separate the two without a known physical reference on each pad.

Timing needs one too. `time_difference_ms` is a reaction time in classic,
adaptive and mirror, and a signed offset from the beat in rhythm. The two are
never pooled here.

## Setup

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")

import pandas as pd

from rehab_analysis import (
    build_catalogue, check, menu_options, prepare, use_style,
    check_selection, keep, need, StaleSelection,
    sec_calibration, sec_overview, sec_quality, sec_compare,
    sec_reaction_time, sec_accuracy, sec_force, sec_individuation,
    sec_rhythm, sec_bilateral, sec_raw, sec_onset, sec_objective_one,
    sec_exclusions, sec_phase, sec_threshold_audit, sec_cue_modality,
    sec_dose, sec_sampling_note, sec_participant_progress,
    sec_summary, write_exports,
)

use_style()

### Check the setup

The packages, and that the sessions folder is where the notebook expects it,
before anything tries to read data.

In [ ]:
ok = check()

## What is on disk

One row per game, oldest first. The id on the left is what the dropdown below
passes along.

In [ ]:
cat = build_catalogue()

if cat.empty:
    print("No recordings on disk yet, so there is nothing to list.")
    print("Play a block in the game, then run this cell again. The cells")
    print("below will all say they have nothing to show rather than fail.")
else:
    print(f"{len(cat)} game(s) across {cat['session'].nunique()} session(s)")

cat[["day", "time", "who", "mode", "hand",
     "trials", "hit_rate", "status"]].rename_axis("id")

## Choose what to analyse

Click one row. Three scopes: one game, one session (a person on one day), or one
person across every day they played.

Choosing only sets the selection. **Run every cell below afterwards, in order.**
Until you choose anything, the newest game is used.

Changing the dropdown and re-running only some of the cells used to blend two
selections into the headline table and the exported CSV without saying so. Each
cell below now checks the dropdown against what was loaded and refuses to run
on a mismatch, so the worst that happens is a message telling you to re-run.

In [ ]:
import ipywidgets as W
from IPython.display import display

pick = "latest"          # what every cell below analyses

note = W.HTML("<span style='color:#64748b'>newest game until you "
              "choose</span>")


def chosen(change):
    global pick
    if change["new"] is None:            # a heading row, not a save
        note.value = ("<span style='color:#b45309'>that row is a heading, "
                      "choose a save under it</span>")
        return
    pick = change["new"]
    note.value = (f"<span style='color:#16a34a'>selected {pick!r}. Now run "
                  f"EVERY cell below, in order, starting with Load the "
                  f"selection.</span>")


menu = W.Dropdown(options=menu_options(cat), value=None,
                  description="Analyse:", layout=W.Layout(width="660px"),
                  style={"description_width": "70px"})
menu.observe(chosen, names="value")
display(W.VBox([menu, note]))

## Load the selection

Reads the trials, the metadata and the calibration once, and adds the calibrated
force columns. Everything below works off what this cell builds, so run it again
after picking something else.

In [ ]:
ctx = prepare(pick)

cat = ctx["cat"]
sel = ctx["sel"]
folders = ctx["folders"]
metas = ctx["metas"]
sessions = ctx["sessions"]
trials = ctx["trials"]
unit = ctx["unit"]
calset = ctx["calset"]
on_task = 0.0            # seconds, filled in by the overview cell below

# What every cell below is now pinned to. They check this against the
# dropdown before touching anything.
print(f"loaded: {ctx['selection']}")

if trials.empty:
    print("\nNothing to analyse in this selection. Every cell below will say")
    print("what it has nothing to show rather than fail, so you can keep")
    print("running them.")
else:
    print(f"{len(folders)} game(s), {len(trials)} trials, "
          f"{sel['session'].nunique()} session(s)")
    print(f"force logged in {unit}, calibration: {calset.status}")

sel[["day", "time", "who", "mode", "hand", "trials"]].rename_axis("id") \
    if not sel.empty else "nothing selected"

## Calibration this data was recorded under

What one light press was worth on each pad on the day, which is the thing that
makes force comparable between fingers. One block per distinct calibration, and
a plain statement when a game recorded none.

In [ ]:
check_selection(ctx, pick)
cal_tables = sec_calibration(metas, sessions, calset)

## Overview

One row per game, then time on task with pauses taken out. `on_task` is reused
by the dose cell further down.

In [ ]:
check_selection(ctx, pick)
on_task = keep(ctx, "on_task", sec_overview(trials, folders, metas))

## Data quality

Cues that never reached the device, how many trials carry force, pauses, and
sensor drift worth looking at.

In [ ]:
check_selection(ctx, pick)
sec_quality(trials, folders, metas)

## Comparing the games

Hit rate, speed and consistency side by side. Prints nothing when the selection
holds a single game.

In [ ]:
check_selection(ctx, pick)
comparison = sec_compare(trials)

## Reaction time

Cued modes only (classic, adaptive, mirror), misses removed. Distribution, per
finger, and the trend across the block.

Rhythm blocks are left out on purpose. Their `time_difference_ms` is a signed
offset from a beat the player already knew was coming, so it is negative about
half the time and it is not a reaction to anything.

In [ ]:
check_selection(ctx, pick)
rt = keep(ctx, "rt", sec_reaction_time(trials))

## Accuracy and the challenge point

Hit rate against the 65 to 80 percent band the adaptive controller aims for,
with missed trials and wrong-finger presses counted separately.

The band belongs to the adaptive controller, so this narrows to adaptive blocks
when the selection has any. The cell prints its scope and the all-cued figure
next to it, because the summary at the bottom exports the all-cued one and the
two are different scopes rather than a disagreement.

In [ ]:
check_selection(ctx, pick)
accuracy = keep(ctx, "accuracy", sec_accuracy(trials))

## Force

Three measures: raw counts as recorded, newtons for the absolute check against
Demouche's healthy data, and force as a fraction of that finger's own
calibration press.

None of the three is a clean between-finger strength comparison, and the cell
says so in full. The fraction divides out the pad, but it divides out the
finger with it, because the reference press is the patient's own light press.
The newtons keep the real differences but keep the pad differences too. This
cell prints both and names the limitation instead of picking one.

In [ ]:
check_selection(ctx, pick)
force = keep(ctx, "force", sec_force(trials, unit, calset))

## Finger individuation

Target-finger force over total force, on two bases.

On absolute readings it is the same basis as the enslavement figures in the
literature (13 percent unimpaired, 25.1 percent after stroke), so that is the
line to put next to them. It carries the per-pad sensitivity bias.

Dividing each lane by its own reference press first removes the pad, but it
weights each finger's spill by the inverse of that finger's own press strength,
so it is **not** comparable with those published figures. The cell prints both
and names each basis, over the same trials, so the difference between them is
the correction and nothing else.

In [ ]:
check_selection(ctx, pick)
ind = keep(ctx, "ind", sec_individuation(trials, calset))

## Rhythm

Beat offsets and whether the tempo was being tracked. Rhythm blocks only.

In [ ]:
check_selection(ctx, pick)
rhythm = keep(ctx, "rhythm", sec_rhythm(trials))

## Both hands

Left against right. Bilateral blocks only. A one-handed selection gets a plain
statement of why there is nothing here rather than a blank cell.

The reaction-time line covers cued trials with a press only. Rhythm beat
offsets and misses are excluded: pooling them gave a left mean of -95 ms and an
"asymmetry" of -8.027 on the shipped data, which was a rhythm block's timing,
not a hand difference.

Each hand is normalised with its own calibration profile. A hand played without
one stays uncorrected and the cell names it, because the two hands sit on eight
different pads.

In [ ]:
check_selection(ctx, pick)
bilateral = keep(ctx, "bilateral", sec_bilateral(trials, unit, calset))

## Raw sample stream

The 200 Hz log behind the first selected game that has one: press durations and
the average shape of a press.

Most selections have no raw.csv, so this usually has nothing to show. It says
which games it looked at and why each had nothing, rather than rendering blank.

In [ ]:
check_selection(ctx, pick)
raw_stream = sec_raw(folders, unit, calset)

## Movement onset and rate of force development

Onset taken from the force trace rather than from a threshold crossing, and how
far apart the two estimates of reaction time sit.

In [ ]:
check_selection(ctx, pick)
onset = keep(ctx, "onset", sec_onset(folders, trials, unit, calset))

## Objective 1, per-finger hit rate

Each finger against the band over its own rolling window of up to 32 trials.
The session-level figure can sit inside the band while single fingers sit well
outside it.

A finger needs 32 trials of its own for one full window. Fingers with fewer are
drawn dashed and left blank in `in_band_share`, because a rolling mean over 8
trials is not the 32-trial block the objective is worded over.

In [ ]:
check_selection(ctx, pick)
objective_one = sec_objective_one(trials, calset=calset)

## Trial exclusions

Trials with no cue delivered and presses faster than 100 ms, with the headline
numbers before and after they come out.

The summary at the bottom is built from the "after exclusions" row. The sections
above print over every recorded trial unless they say otherwise, so a figure
there can differ from the summary, and this table is where to see by how much.

In [ ]:
check_selection(ctx, pick)
flagged = keep(ctx, "flagged", sec_exclusions(trials))

## Pretest to aftertest

Only prints once a protocol with phases has been run.

In [ ]:
check_selection(ctx, pick)
phases = sec_phase(trials)

## Press thresholds in newtons

What force each finger needed to register a press, against the healthy
fingertip forces Demouche measured. A trigger above those is a threshold
problem, not a weak finger.

In [ ]:
check_selection(ctx, pick)
thresholds = sec_threshold_audit(metas=metas, calset=calset)

## Cue modality

Visual, vibration and both compared. Needs blocks recorded under at least two
cue settings.

In [ ]:
check_selection(ctx, pick)
cues = sec_cue_modality(trials, calset)

## Dose

Repetitions against Lang's clinical benchmark. Needs `on_task` from the overview
cell above. Without it the per-minute lines are skipped rather than guessed at.

In [ ]:
check_selection(ctx, pick)
on_task = need(ctx, "on_task")["on_task"]
sec_dose(trials, on_task / 60)

## Sampling

How many logged samples carry new sensor data. That sets the real resolution of
every onset and rate-of-force figure, so it belongs in the limitations.

In [ ]:
check_selection(ctx, pick)
sampling = sec_sampling_note(folders)

## Progress per participant

Every session a person has done, in order. This covers everyone on disk, not
just the selection, because the trend across sessions is the outcome measure.

A **session** is one person on one day. Two blocks in one sitting are one row,
not two: counting games as sessions turned a single sitting into a training
trend.

In [ ]:
check_selection(ctx, pick)
progress = sec_participant_progress(cat=cat)

## Headline numbers

Built from the trials that can be analysed, which is not the same as every
trial recorded. A trial whose cue command never reached the device was never
presented, and a press under 100 ms is anticipation rather than a response. The
table names the counts so the basis is visible.

`need` below refuses to run unless every cell above was run for the selection
the dropdown currently names, so this table cannot be assembled out of two
different saves.

In [ ]:
check_selection(ctx, pick)

# Every piece this table leans on has to have been computed for THIS
# selection. Without the check, re-running only some cells after changing
# the dropdown built the table out of two different saves and said
# nothing about it.
ran = need(ctx, "on_task", "rt", "accuracy", "force", "ind", "rhythm",
           "onset", "flagged")

summary = sec_summary(trials, unit, calset, ran["on_task"],
                      folders=folders, onset=ran["onset"],
                      accuracy=ran["accuracy"])
keep(ctx, "summary", summary)

pd.DataFrame([summary]).T.rename(columns={0: "value"})

## Save the tables

Writes the CSVs next to this notebook. Skip this cell to leave the files on disk
as they are.

`selected_trials.csv` keeps every recorded trial and adds `excluded` and
`exclusion_reason` to each row, so anyone recomputing from the CSV can land on
the same figures as the summary by filtering on `excluded == False`.

In [ ]:
check_selection(ctx, pick)
ran = need(ctx, "summary", "ind")

write_exports(ran["summary"], trials, calset, ran["ind"])